
# Agentic Multi-Worker — LangGraph Diagnosis Notebook

This notebook is **self-contained**: it defines a minimal multi-agent pipeline using **LangGraph** with a Supervisor → Tools → Worker flow.
You can run it with a tiny local model (e.g., **Ollama** `qwen2.5:1.5b-instruct` or `llama3.2:1b`) and later switch to your 7B via **vLLM**.



> **Setup (run locally):**  
> 1. Start a backend (pick one)  
>    - **Ollama (dev 1B)**:  
>      ```bash
>      ollama pull qwen2.5:1.5b-instruct
>      ollama serve
>      ```
>    - **vLLM (7B later)**:  
>      ```bash
>      python -m vllm.entrypoints.openai.api_server --model <your-7b> --host 0.0.0.0 --port 8000
>      ```
> 2. Install deps (in your env, *not* here):  
>    ```bash
>    pip install requests langgraph langchain-core
>    ```


In [17]:

# === Config ===
from dataclasses import dataclass

@dataclass
class LLMConfig:
    provider: str         # "ollama" | "vllm"
    model: str            # e.g., "qwen2.5:1.5b-instruct" or "your-7b"
    endpoint: str | None  # e.g., "http://localhost:11434" or "http://localhost:8000/v1"
    temperature: float = 0.2
    top_p: float = 0.9
    max_tokens: int = 1024
    json_mode: bool = True
    num_ctx: int = 8192   # for Ollama

# Choose your backend here:
CFG = LLMConfig(provider="ollama", model="qwen2.5:1.5b-instruct", endpoint="http://localhost:11434")
# For vLLM later:
# CFG = LLMConfig(provider="vllm", model="your-7b-checkpoint", endpoint="http://localhost:8000/v1")


In [18]:
CFG

LLMConfig(provider='ollama', model='qwen2.5:1.5b-instruct', endpoint='http://localhost:11434', temperature=0.2, top_p=0.9, max_tokens=1024, json_mode=True, num_ctx=8192)

In [19]:
# === LLM Adapter (unified for Ollama + vLLM) ===
import requests

class LLM:
    def __init__(self, cfg: LLMConfig):
        self.cfg = cfg

    def chat(self, messages, tools=None, tool_choice="auto"):
        p = self.cfg.provider
        if p == "ollama":
            payload = {
                "model": self.cfg.model,
                "messages": messages,
                "stream": False,
                "options": {"temperature": self.cfg.temperature, "num_ctx": self.cfg.num_ctx},
                "format": "json" if self.cfg.json_mode else None
            }
            if tools: 
                payload["tools"] = tools
            r = requests.post(f"{self.cfg.endpoint}/api/chat", json=payload, timeout=120)
            r.raise_for_status()
            return r.json()["message"]  # {role, content, tool_calls?}

        if p == "vllm":
            payload = {
                "model": self.cfg.model,
                "messages": messages,
                "temperature": self.cfg.temperature,
                "top_p": self.cfg.top_p,
                "max_tokens": self.cfg.max_tokens,
                "tools": tools or [],
                "tool_choice": tool_choice,
                "response_format": {"type": "json_object"} if self.cfg.json_mode else {"type": "text"},
            }
            r = requests.post(f"{self.cfg.endpoint}/chat/completions", json=payload, timeout=120)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]

        raise ValueError(f"Unknown provider {p}")

In [20]:

# === Tool specs and runtime stubs ===

def tool_specs():
    return [
      {
        "type": "function",
        "function": {
          "name": "search_guidelines",
          "description": "Search vetted clinical guidelines; return citations.",
          "parameters": {
            "type": "object",
            "properties": {
              "query": {"type": "string"},
              "section": {"type": "string", "enum": ["diagnosis","treatment","monitoring","drug-dosing"]}
            },
            "required": ["query"]
          }
        }
      },
      {
        "type": "function",
        "function": {
          "name": "retrieve_rag",
          "description": "Retrieve top-k passages from local medical corpus.",
          "parameters": {
            "type": "object",
            "properties": {"query":{"type":"string"}, "k":{"type":"integer","default":5}},
            "required": ["query"]
          }
        }
      },
      {
        "type": "function",
        "function": {
          "name": "calc_dose",
          "description": "Performs basic medication dose calculations.",
          "parameters": {
            "type": "object",
            "properties": {"rule":{"type":"string"}, "weight_kg":{"type":"number"}, "egfr":{"type":"number"}},
            "required": ["rule","weight_kg"]
          }
        }
      },
      {
        "type": "function",
        "function": {
          "name": "summarize_soap",
          "description": "Format structured SOAP note from inputs and citations.",
          "parameters": {
            "type": "object",
            "properties": {"subjective":{"type":"string"},"objective":{"type":"string"},"assessment":{"type":"string"},"plan":{"type":"string"}},
            "required": ["subjective","objective","assessment","plan"]
          }
        }
      }
    ]

# Stubs — replace with your real RAG/guideline sources
def search_guidelines(query, section=None):
    return {"hits":[{"title":"ESC 2024 Hypertension","section": section or "diagnosis", "id":"esc24-htn-1"}]}

def retrieve_rag(query, k=5):
    return {"passages":[{"id":"pmid12345","text":"ACE inhibitor is indicated in HTN with diabetes and albuminuria."}]}

def calc_dose(rule, weight_kg, egfr=None):
    dose = round(weight_kg * 2.0, 1)
    return {"dose_mg": dose, "notes": f"Rule={rule}, EGFR={egfr}"}

def summarize_soap(subjective, objective, assessment, plan):
    return {"soap":{"S":subjective,"O":objective,"A":assessment,"P":plan}}

TOOL_IMPL = {
    "search_guidelines": search_guidelines,
    "retrieve_rag": retrieve_rag,
    "calc_dose": calc_dose,
    "summarize_soap": summarize_soap
}


In [21]:
# === Prompts ===
SUPERVISOR_SYS = """You are the Supervisor. Decide which worker should handle the query and which tools to call.

Workers: cardiology, hypertension, diabetes
Tools: search_guidelines, retrieve_rag, calc_dose, summarize_soap

Respond with valid JSON only (no markdown, no explanation):
{"worker": "hypertension", "plan": [{"tool":"search_guidelines","args":{"query":"hypertension treatment", "section":"treatment"}}]}"""

WORKER_SYS = """You are the {domain} specialist. Use the TOOL_RESULTS provided to answer the question.
Never invent facts. Cite sources using their IDs.

Respond with valid JSON only (no markdown, no explanation):
{{"answer": "your answer here", "citations": ["id1", "id2"], "confidence": 0.8}}"""

In [22]:
# === Supervisor / Worker wrappers ===
import json

def make_supervisor_plan(llm: LLM, user_msg: str):
    messages = [{"role":"system","content": SUPERVISOR_SYS},
                {"role":"user","content": user_msg}]
    res = llm.chat(messages=messages, tools=tool_specs())
    content = res.get("content", "{}")
    
    # Debug output
    print(f"DEBUG - Raw supervisor response:\n{repr(content)}\n")
    
    # Clean up content
    if content:
        if content.startswith("```"):
            lines = content.split("\n")
            content = "\n".join(lines[1:-1] if len(lines) > 2 else lines)
            content = content.strip()
        if content.startswith("json"):
            content = content[4:].strip()
    
    # Try parsing content first
    try:
        plan = json.loads(content)
        if "worker" in plan and "plan" in plan:
            print(f"DEBUG - Parsed plan from content: {plan}\n")
            return plan
    except (json.JSONDecodeError, TypeError) as e:
        print(f"DEBUG - JSON parse error: {e}")
    
    # Fallback: derive from tool_calls if available
    plan = {"worker": "hypertension", "plan": []}
    tool_calls = res.get("tool_calls", [])
    if tool_calls:
        print(f"DEBUG - Using tool_calls fallback: {tool_calls}\n")
        for tc in tool_calls:
            func = tc.get("function", {})
            args = func.get("arguments", {})
            if isinstance(args, str):
                try:
                    args = json.loads(args)
                except:
                    args = {}
            plan["plan"].append({"tool": func.get("name", ""), "args": args})
    else:
        print("DEBUG - No tool_calls found, using default plan\n")
        # Provide a sensible default
        plan["plan"] = [
            {"tool": "search_guidelines", "args": {"query": user_msg, "section": "treatment"}},
            {"tool": "retrieve_rag", "args": {"query": user_msg}}
        ]
    
    return plan

def run_worker(llm: LLM, domain: str, user_msg: str, trace: list[dict]):
    sys_prompt = WORKER_SYS.format(domain=domain)
    messages = [
        {"role":"system","content": sys_prompt},
        {"role":"user","content": user_msg},
        {"role":"user","content": "TOOL_RESULTS:\n" + json.dumps(trace, ensure_ascii=False)}
    ]
    res = llm.chat(messages=messages)
    content = res.get("content","{}").strip()
    
    # Debug: print raw content
    print(f"DEBUG - Raw worker response:\n{repr(content)}\n")
    
    # Try to clean up common JSON issues
    if content:
        # Remove markdown code blocks if present
        if content.startswith("```"):
            lines = content.split("\n")
            content = "\n".join(lines[1:-1] if len(lines) > 2 else lines)
            content = content.strip()
        
        # Remove "json" language identifier
        if content.startswith("json"):
            content = content[4:].strip()
    
    try:
        parsed = json.loads(content)
        return parsed
    except (json.JSONDecodeError, TypeError) as e:
        print(f"DEBUG - JSON parse error: {e}")
        print(f"DEBUG - Failed content: {repr(content)}")
        # Return a safe fallback
        return {
            "answer": content if content else "Unable to generate response", 
            "citations": [], 
            "confidence": 0.2
        }

In [23]:

# === LangGraph StateGraph ===
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END

class GraphState(TypedDict, total=False):
    user_msg: str
    plan: Dict[str, Any]
    trace: List[Dict[str, Any]]
    final: Dict[str, Any]

def plan_node(state: GraphState, llm: LLM) -> GraphState:
    plan = make_supervisor_plan(llm, state["user_msg"])
    return {"plan": plan}

def tools_node(state: GraphState, llm: LLM) -> GraphState:
    plan = state["plan"]
    trace = []
    for step in plan.get("plan", []):
        name = step["tool"]
        args = step.get("args", {}) if isinstance(step.get("args"), dict) else {}
        out = TOOL_IMPL[name](**args)
        trace.append({"tool": name, "args": args, "output": out})
    return {"trace": trace}

def worker_node(state: GraphState, llm: LLM) -> GraphState:
    plan = state["plan"]
    final = run_worker(llm, plan.get("worker","hypertension"), state["user_msg"], state.get("trace", []))
    return {"final": final}

def build_graph(llm: LLM):
    g = StateGraph(GraphState)
    g.add_node("plan", lambda s: plan_node(s, llm))
    g.add_node("tools", lambda s: tools_node(s, llm))
    g.add_node("worker", lambda s: worker_node(s, llm))

    g.set_entry_point("plan")
    g.add_edge("plan", "tools")
    g.add_edge("tools", "worker")
    g.add_edge("worker", END)

    return g.compile()

def run_graph(llm: LLM, user_msg: str) -> Dict[str, Any]:
    app = build_graph(llm)
    out = app.invoke({"user_msg": user_msg})
    return {"plan": out.get("plan"), "trace": out.get("trace"), "final": out.get("final")}


## Run a sample query

In [24]:

llm = LLM(CFG)
query = "New HTN dx in T2D, eGFR 55—first-line and dose?"
result = run_graph(llm, query)
result.keys()


DEBUG - Raw supervisor response:
''

DEBUG - JSON parse error: Expecting value: line 1 column 1 (char 0)
DEBUG - Using tool_calls fallback: [{'function': {'name': 'calc_dose', 'arguments': {'rule': 'first_line', 'weight_kg': 60}}}, {'function': {'name': 'search_guidelines', 'arguments': {'query': 'hypertension treatment', 'section': 'treatment'}}}]

DEBUG - Raw worker response:
'{\n    "answer": "The first-line treatment for hypertension in patients with type 2 diabetes (T2D) and an estimated glomerular filtration rate (eGFR) of 55 mL/min/1.73m² is amlodipine besylate at a dose of 120 mg once daily.",\n    "citations": ["id1", "esc24-htn-1"],\n    "confidence": 0.8\n}'



dict_keys(['plan', 'trace', 'final'])

In [ ]:

import json
print("PLAN:\n", json.dumps(result["plan"], indent=2))
print("\nTRACE:\n", json.dumps(result["trace"], indent=2))
print("\nFINAL:\n", json.dumps(result["final"], indent=2))


## Helper: run multiple queries

In [ ]:

def run_queries(queries: list[str]):
    out = []
    for q in queries:
        out.append({"q": q, **run_graph(LLM(CFG), q)})
    return out

# Example:
# batch = run_queries([
#     "Hypertension with T2D and albuminuria — preferred class?",
#     "Stable angina with HTN — beta-blocker role?",
# ])
# batch


In [25]:
# === Config ===
from dataclasses import dataclass

@dataclass
class LLMConfig:
    provider: str         # "ollama" | "vllm"
    model: str            # e.g., "qwen2.5:1.5b-instruct" or "your-7b"
    endpoint: str | None  # e.g., "http://localhost:11434" or "http://localhost:8000/v1"
    temperature: float = 0.2
    top_p: float = 0.9
    max_tokens: int = 1024
    json_mode: bool = True
    num_ctx: int = 8192   # for Ollama

# Choose your backend here:
CFG = LLMConfig(provider="ollama", model="qwen2.5:1.5b-instruct", endpoint="http://localhost:11434")
# For vLLM later:
# CFG = LLMConfig(provider="vllm", model="your-7b-checkpoint", endpoint="http://localhost:8000/v1")


# === LLM Adapter (unified for Ollama + vLLM) ===
import requests
import json

class LLM:
    """Unified LLM interface for Ollama and vLLM"""
    def __init__(self, cfg: LLMConfig, domain: str = None):
        self.cfg = cfg
        self.domain = domain  # For future: load domain-specific adapter
    
    def chat(self, messages, tools=None, tool_choice="auto"):
        p = self.cfg.provider
        if p == "ollama":
            payload = {
                "model": self.cfg.model,
                "messages": messages,
                "stream": False,
                "options": {"temperature": self.cfg.temperature, "num_ctx": self.cfg.num_ctx},
                "format": "json" if self.cfg.json_mode else None
            }
            if tools: 
                payload["tools"] = tools
            
            try:
                r = requests.post(f"{self.cfg.endpoint}/api/chat", json=payload, timeout=120)
                r.raise_for_status()
                return r.json()["message"]
            except Exception as e:
                print(f"ERROR - Ollama request failed: {e}")
                return {"role": "assistant", "content": "{}"}

        if p == "vllm":
            payload = {
                "model": self.cfg.model,
                "messages": messages,
                "temperature": self.cfg.temperature,
                "top_p": self.cfg.top_p,
                "max_tokens": self.cfg.max_tokens,
                "tools": tools or [],
                "tool_choice": tool_choice,
                "response_format": {"type": "json_object"} if self.cfg.json_mode else {"type": "text"},
            }
            
            try:
                r = requests.post(f"{self.cfg.endpoint}/chat/completions", json=payload, timeout=120)
                r.raise_for_status()
                return r.json()["choices"][0]["message"]
            except Exception as e:
                print(f"ERROR - vLLM request failed: {e}")
                return {"role": "assistant", "content": "{}"}

        raise ValueError(f"Unknown provider {p}")


# === Tool specs and runtime stubs ===

def tool_specs():
    """Return OpenAI-compatible tool specifications"""
    return [
      {
        "type": "function",
        "function": {
          "name": "search_guidelines",
          "description": "Search vetted clinical guidelines; return citations.",
          "parameters": {
            "type": "object",
            "properties": {
              "query": {"type": "string", "description": "Search query for guidelines"},
              "section": {
                "type": "string", 
                "enum": ["diagnosis","treatment","monitoring","drug-dosing"],
                "description": "Specific section to search"
              }
            },
            "required": ["query"]
          }
        }
      },
      {
        "type": "function",
        "function": {
          "name": "retrieve_rag",
          "description": "Retrieve top-k passages from local medical corpus.",
          "parameters": {
            "type": "object",
            "properties": {
              "query": {"type": "string", "description": "Search query"},
              "k": {"type": "integer", "default": 5, "description": "Number of passages to retrieve"}
            },
            "required": ["query"]
          }
        }
      },
      {
        "type": "function",
        "function": {
          "name": "calc_dose",
          "description": "Performs basic medication dose calculations.",
          "parameters": {
            "type": "object",
            "properties": {
              "rule": {"type": "string", "description": "Dosing rule or medication name"},
              "weight_kg": {"type": "number", "description": "Patient weight in kg"},
              "egfr": {"type": "number", "description": "eGFR value for renal dosing"}
            },
            "required": ["rule", "weight_kg"]
          }
        }
      },
      {
        "type": "function",
        "function": {
          "name": "check_interactions",
          "description": "Check for drug-drug interactions and contraindications.",
          "parameters": {
            "type": "object",
            "properties": {
              "medications": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of medications to check"
              },
              "conditions": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Patient conditions"
              }
            },
            "required": ["medications"]
          }
        }
      }
    ]

# Tool implementations (stubs - replace with real implementations)
def search_guidelines(query: str, section: str = None):
    """Stub: Search clinical guidelines"""
    return {
        "hits": [
            {
                "title": "AHA/ACC 2024 Hypertension Guidelines",
                "section": section or "treatment",
                "id": "aha-acc-2024-htn",
                "snippet": "ACE inhibitors or ARBs are first-line for HTN with diabetes and albuminuria."
            },
            {
                "title": "ESC 2024 Guidelines",
                "section": section or "treatment", 
                "id": "esc-2024-htn",
                "snippet": "Consider CKD stage when selecting antihypertensive."
            }
        ]
    }

def retrieve_rag(query: str, k: int = 5):
    """Stub: RAG retrieval from medical corpus"""
    return {
        "passages": [
            {
                "id": "pmid12345",
                "text": "ACE inhibitor is indicated in HTN with diabetes and albuminuria. Adjust dose for eGFR <60.",
                "relevance": 0.92
            },
            {
                "id": "pmid67890",
                "text": "First-line for HTN in T2D: ACE-I or ARB. Monitor potassium and creatinine.",
                "relevance": 0.88
            }
        ]
    }

def calc_dose(rule: str, weight_kg: float, egfr: float = None):
    """Stub: Dose calculation"""
    base_dose = round(weight_kg * 2.0, 1)
    
    # Adjust for renal function
    if egfr and egfr < 60:
        base_dose = base_dose * 0.75
        note = f"Dose reduced for eGFR {egfr}"
    else:
        note = "Standard dosing"
    
    return {
        "dose_mg": round(base_dose, 1),
        "notes": f"Rule={rule}, eGFR={egfr}, {note}",
        "frequency": "once daily"
    }

def check_interactions(medications: list[str], conditions: list[str] = None):
    """Stub: Check drug interactions"""
    return {
        "interactions": [
            {"severity": "moderate", "description": "ACE-I + ARB: increased hyperkalemia risk"}
        ],
        "contraindications": [],
        "warnings": ["Monitor renal function and potassium"]
    }

# Tool registry
TOOL_IMPL = {
    "search_guidelines": search_guidelines,
    "retrieve_rag": retrieve_rag,
    "calc_dose": calc_dose,
    "check_interactions": check_interactions
}


# === Prompts ===

ROUTER_SYS = """You are a medical triage coordinator. Analyze the clinical query and determine:
1. Which specialist(s) should be consulted: cardiology, diabetes, hypertension
2. Whether this requires multiple specialists (co-morbidity cases)
3. Clinical urgency

Respond with valid JSON only (no markdown):
{
  "primary": "cardiology|diabetes|hypertension",
  "consult": ["other_domain1", "other_domain2"],
  "requires_consensus": true,
  "urgency": "routine|urgent|emergent",
  "reasoning": "brief explanation"
}"""

SPECIALIST_SYS = """You are a {domain} specialist in a multi-disciplinary medical team.

Your role:
1. Analyze the clinical query from your specialty's perspective
2. Use available tools to gather evidence
3. Flag any concerns or contraindications relevant to your domain
4. Consider input from other specialists if provided
5. Never invent facts - only use tool results and established guidelines

Respond with valid JSON only (no markdown):
{{
  "assessment": "your clinical assessment",
  "recommendations": ["rec1", "rec2"],
  "concerns": ["concern1", "concern2"],
  "tool_needs": ["tool1", "tool2"],
  "contraindications": [],
  "confidence": 0.8,
  "citations": ["id1", "id2"]
}}"""

CONSENSUS_SYS = """You are a clinical coordinator synthesizing multi-specialist input.

Your role:
1. Reconcile recommendations from different specialists
2. Identify conflicts or contraindications
3. Prioritize actions based on urgency and safety
4. Create a unified, actionable care plan
5. Highlight areas of agreement vs disagreement

Respond with valid JSON only (no markdown):
{
  "final_assessment": "integrated clinical picture",
  "unified_plan": ["step1", "step2", "step3"],
  "critical_alerts": ["alert1"],
  "specialist_agreement": "high|medium|low",
  "conflicts_resolved": ["conflict and resolution"],
  "confidence": 0.85,
  "next_steps": ["action1", "action2"]
}"""


# === Helper Functions ===

def safe_json_parse(content: str, default: dict = None) -> dict:
    """Safely parse JSON with fallback"""
    if default is None:
        default = {}
    
    if not content:
        return default
    
    # Clean up common JSON issues
    content = content.strip()
    
    # Remove markdown code blocks
    if content.startswith("```"):
        lines = content.split("\n")
        content = "\n".join(lines[1:-1] if len(lines) > 2 else lines)
        content = content.strip()
    
    # Remove language identifier
    if content.startswith("json"):
        content = content[4:].strip()
    
    try:
        return json.loads(content)
    except (json.JSONDecodeError, TypeError) as e:
        print(f"⚠️  JSON parse error: {e}")
        print(f"   Content: {repr(content[:200])}")
        return {**default, "_raw_content": content}


def execute_tools(tool_requests: list[dict]) -> list[dict]:
    """Execute requested tools and return results"""
    results = []
    
    for req in tool_requests:
        tool_name = req.get("tool")
        args = req.get("args", {})
        
        if tool_name not in TOOL_IMPL:
            results.append({
                "tool": tool_name,
                "args": args,
                "output": {"error": f"Unknown tool: {tool_name}"}
            })
            continue
        
        try:
            output = TOOL_IMPL[tool_name](**args)
            results.append({
                "tool": tool_name,
                "args": args,
                "output": output
            })
        except Exception as e:
            results.append({
                "tool": tool_name,
                "args": args,
                "output": {"error": str(e)}
            })
    
    return results


# === Agent Nodes ===

def router_node(state: dict, llm: LLM) -> dict:
    """Routes query to appropriate specialists"""
    print("\n🔀 ROUTER: Analyzing query...")
    
    messages = [
        {"role": "system", "content": ROUTER_SYS},
        {"role": "user", "content": state["user_msg"]}
    ]
    
    res = llm.chat(messages)
    content = res.get("content", "{}")
    
    routing = safe_json_parse(content, default={
        "primary": "hypertension",
        "consult": ["cardiology", "diabetes"],
        "requires_consensus": True,
        "urgency": "routine",
        "reasoning": "Default routing - all specialists consulted"
    })
    
    print(f"   Primary: {routing.get('primary')}")
    print(f"   Consult: {routing.get('consult', [])}")
    print(f"   Consensus needed: {routing.get('requires_consensus')}")
    
    return {"routing": routing}


def specialist_node(domain: str, state: dict, llm: LLM) -> dict:
    """Generic specialist agent - returns response for given domain"""
    
    # Check if this specialist should run
    routing = state.get("routing", {})
    active_domains = [routing.get("primary")] + routing.get("consult", [])
    
    if domain not in active_domains:
        print(f"   ⏭️  {domain.upper()}: Skipped (not in routing)")
        return {}
    
    print(f"\n🏥 {domain.upper()} SPECIALIST: Analyzing...")
    
    # Gather context from other specialists
    context_parts = []
    for other_domain in ["cardiology", "diabetes", "hypertension"]:
        if other_domain == domain:
            continue
        resp = state.get(f"{other_domain}_response", {})
        if resp and "assessment" in resp:
            context_parts.append(f"{other_domain.title()}: {resp.get('assessment', '')}")
    
    context_str = "\n".join(context_parts) if context_parts else "No input from other specialists yet."
    
    # Build specialist prompt
    prompt = f"""Clinical Query: {state['user_msg']}

Other Specialists' Input:
{context_str}

Tool Results Available:
{json.dumps(state.get('tool_results', []), indent=2)}

Provide your {domain} specialist assessment."""
    
    messages = [
        {"role": "system", "content": SPECIALIST_SYS.format(domain=domain)},
        {"role": "user", "content": prompt}
    ]
    
    res = llm.chat(messages, tools=tool_specs())
    content = res.get("content", "{}")
    
    response = safe_json_parse(content, default={
        "assessment": f"Assessment from {domain} specialist",
        "recommendations": [],
        "concerns": [],
        "confidence": 0.5
    })
    
    # Execute any new tools requested
    if response.get("tool_needs"):
        print(f"   🔧 Executing tools: {response.get('tool_needs')}")
        new_tools = [{"tool": t, "args": {"query": state["user_msg"]}} for t in response["tool_needs"]]
        tool_results = execute_tools(new_tools)
        
        # Update state with new tool results
        existing_results = state.get("tool_results", [])
        state["tool_results"] = existing_results + tool_results
    
    print(f"   Confidence: {response.get('confidence', 0):.2f}")
    print(f"   Recommendations: {len(response.get('recommendations', []))}")
    
    return response


def tools_node(state: dict, llm: LLM) -> dict:
    """Execute initial tool calls based on routing"""
    print("\n🔧 TOOLS: Gathering initial evidence...")
    
    routing = state.get("routing", {})
    query = state["user_msg"]
    
    # Default tool plan
    tool_plan = [
        {"tool": "search_guidelines", "args": {"query": query, "section": "treatment"}},
        {"tool": "retrieve_rag", "args": {"query": query, "k": 3}}
    ]
    
    # Execute tools
    results = execute_tools(tool_plan)
    
    print(f"   Executed {len(results)} tools")
    
    return {"tool_results": results}


def consensus_node(state: dict, llm: LLM) -> dict:
    """Synthesize all specialist inputs into final recommendation"""
    print("\n🤝 CONSENSUS: Synthesizing recommendations...")
    
    # Gather all specialist responses
    specialists = {
        "cardiology": state.get("cardiology_response", {}),
        "diabetes": state.get("diabetes_response", {}),
        "hypertension": state.get("hypertension_response", {})
    }
    
    # Filter to only specialists who responded
    active_specialists = {k: v for k, v in specialists.items() if v}
    
    prompt = f"""Clinical Query: {state['user_msg']}

Specialist Assessments:
{json.dumps(active_specialists, indent=2)}

Tool Evidence:
{json.dumps(state.get('tool_results', []), indent=2)}

Synthesize into unified clinical recommendation."""
    
    messages = [
        {"role": "system", "content": CONSENSUS_SYS},
        {"role": "user", "content": prompt}
    ]
    
    res = llm.chat(messages)
    content = res.get("content", "{}")
    
    consensus = safe_json_parse(content, default={
        "final_assessment": "Consensus assessment",
        "unified_plan": [],
        "confidence": 0.5
    })
    
    print(f"   Agreement level: {consensus.get('specialist_agreement', 'unknown')}")
    print(f"   Critical alerts: {len(consensus.get('critical_alerts', []))}")
    
    return {
        "consensus": consensus,
        "final": consensus
    }


# === LangGraph Construction ===

from typing import TypedDict, Dict, Any, List
from langgraph.graph import StateGraph, END

class CollabState(TypedDict, total=False):
    user_msg: str
    routing: Dict[str, Any]
    tool_results: List[Dict[str, Any]]
    cardiology_response: Dict[str, Any]
    diabetes_response: Dict[str, Any]
    hypertension_response: Dict[str, Any]
    consensus: Dict[str, Any]
    final: Dict[str, Any]


def build_collaborative_graph(llm: LLM):
    """Build the collaborative multi-agent graph"""
    
    g = StateGraph(CollabState)
    
    # Add nodes
    g.add_node("router", lambda s: router_node(s, llm))
    g.add_node("tools", lambda s: tools_node(s, llm))
    g.add_node("cardiology", lambda s: {"cardiology_response": specialist_node("cardiology", s, llm)})
    g.add_node("diabetes", lambda s: {"diabetes_response": specialist_node("diabetes", s, llm)})
    g.add_node("hypertension", lambda s: {"hypertension_response": specialist_node("hypertension", s, llm)})
    g.add_node("consensus", lambda s: consensus_node(s, llm))
    
    # Define flow
    g.set_entry_point("router")
    g.add_edge("router", "tools")
    
    # After tools, all specialists run in parallel
    g.add_edge("tools", "cardiology")
    g.add_edge("tools", "diabetes")
    g.add_edge("tools", "hypertension")
    
    # All specialists feed into consensus
    g.add_edge("cardiology", "consensus")
    g.add_edge("diabetes", "consensus")
    g.add_edge("hypertension", "consensus")
    
    g.add_edge("consensus", END)
    
    return g.compile()


def run_collaborative_graph(llm: LLM, user_msg: str) -> Dict[str, Any]:
    """Run the collaborative multi-agent system"""
    print(f"\n{'='*70}")
    print(f"🚀 COLLABORATIVE MULTI-AGENT SYSTEM")
    print(f"{'='*70}")
    print(f"Query: {user_msg}")
    
    app = build_collaborative_graph(llm)
    result = app.invoke({"user_msg": user_msg})
    
    print(f"\n{'='*70}")
    print(f"✅ COMPLETE")
    print(f"{'='*70}\n")
    
    return result


# === Convenience Functions ===

def print_result(result: Dict[str, Any]):
    """Pretty print the result"""
    print("\n" + "="*70)
    print("📋 ROUTING DECISION")
    print("="*70)
    routing = result.get("routing", {})
    print(f"Primary: {routing.get('primary')}")
    print(f"Consult: {routing.get('consult', [])}")
    print(f"Reasoning: {routing.get('reasoning', 'N/A')}")
    
    print("\n" + "="*70)
    print("🏥 SPECIALIST ASSESSMENTS")
    print("="*70)
    
    for domain in ["cardiology", "diabetes", "hypertension"]:
        resp = result.get(f"{domain}_response", {})
        if resp:
            print(f"\n{domain.upper()}:")
            print(f"  Assessment: {resp.get('assessment', 'N/A')[:150]}...")
            print(f"  Confidence: {resp.get('confidence', 0):.2f}")
            if resp.get('recommendations'):
                print(f"  Recommendations: {resp.get('recommendations')}")
    
    print("\n" + "="*70)
    print("🤝 CONSENSUS")
    print("="*70)
    consensus = result.get("consensus", {})
    print(f"Final Assessment: {consensus.get('final_assessment', 'N/A')}")
    print(f"Agreement Level: {consensus.get('specialist_agreement', 'N/A')}")
    print(f"Confidence: {consensus.get('confidence', 0):.2f}")
    
    if consensus.get("unified_plan"):
        print("\nUnified Plan:")
        for i, step in enumerate(consensus.get("unified_plan", []), 1):
            print(f"  {i}. {step}")
    
    if consensus.get("critical_alerts"):
        print("\n⚠️  CRITICAL ALERTS:")
        for alert in consensus.get("critical_alerts", []):
            print(f"  • {alert}")


def run_batch_queries(queries: list[str], llm: LLM = None):
    """Run multiple queries in batch"""
    if llm is None:
        llm = LLM(CFG)
    
    results = []
    for i, query in enumerate(queries, 1):
        print(f"\n\n{'#'*70}")
        print(f"QUERY {i}/{len(queries)}")
        print(f"{'#'*70}")
        
        result = run_collaborative_graph(llm, query)
        results.append({"query": query, "result": result})
    
    return results

In [26]:
# === Example Usage ===

# Create LLM instance
llm = LLM(CFG)
llm

In [28]:
# Single query
query = "58yo male, new HTN diagnosis with T2D (A1c 8.5%) and eGFR 55. What's first-line therapy and appropriate dose?"
result = run_collaborative_graph(llm, query)


🚀 COLLABORATIVE MULTI-AGENT SYSTEM
Query: 58yo male, new HTN diagnosis with T2D (A1c 8.5%) and eGFR 55. What's first-line therapy and appropriate dose?

🔀 ROUTER: Analyzing query...
   Primary: ['cardiology', 'diabetes']
   Consult: []
   Consensus needed: False

🔧 TOOLS: Gathering initial evidence...
   Executed 2 tools
   ⏭️  CARDIOLOGY: Skipped (not in routing)
   ⏭️  DIABETES: Skipped (not in routing)
   ⏭️  HYPERTENSION: Skipped (not in routing)

🤝 CONSENSUS: Synthesizing recommendations...
   Agreement level: high
   Critical alerts: 0

✅ COMPLETE



In [29]:
# Pretty print
print_result(result)

# Access specific parts
print("\n" + "="*70)
print("📊 RAW DATA ACCESS")
print("="*70)
print(f"Routing: {result.get('routing')}")
print(f"Tool results: {len(result.get('tool_results', []))} tools executed")
print(f"Final confidence: {result.get('final', {}).get('confidence', 0):.2f}")


📋 ROUTING DECISION
Primary: ['cardiology', 'diabetes']
Consult: []
Reasoning: The patient presents with newly diagnosed hypertension, type 2 diabetes mellitus (with an A1C of 8.5%), and a reduced estimated glomerular filtration rate (eGFR of 55). The first-line therapy for these conditions typically includes lifestyle modifications such as diet and exercise, followed by pharmacological interventions.

🏥 SPECIALIST ASSESSMENTS

🤝 CONSENSUS
Final Assessment: The patient's hypertension management should be guided by the AHA/ACC 2024 Hypertension Guidelines, which recommend ACE inhibitors or ARBs as first-line therapy for HTN with diabetes and albuminuria.
Agreement Level: high
Confidence: 0.85

Unified Plan:
  1. Start ACE inhibitor or ARB
  2. Monitor eGFR and adjust dose if necessary

📊 RAW DATA ACCESS
Routing: {'primary': ['cardiology', 'diabetes'], 'consult': [], 'requires_consensus': False, 'urgency': 'routine', 'reasoning': 'The patient presents with newly diagnosed hypertension, t

In [30]:
result.get('tool_results', [])

[{'tool': 'search_guidelines',
  'args': {'query': "58yo male, new HTN diagnosis with T2D (A1c 8.5%) and eGFR 55. What's first-line therapy and appropriate dose?",
   'section': 'treatment'},
  'output': {'hits': [{'title': 'AHA/ACC 2024 Hypertension Guidelines',
     'section': 'treatment',
     'id': 'aha-acc-2024-htn',
     'snippet': 'ACE inhibitors or ARBs are first-line for HTN with diabetes and albuminuria.'},
    {'title': 'ESC 2024 Guidelines',
     'section': 'treatment',
     'id': 'esc-2024-htn',
     'snippet': 'Consider CKD stage when selecting antihypertensive.'}]}},
 {'tool': 'retrieve_rag',
  'args': {'query': "58yo male, new HTN diagnosis with T2D (A1c 8.5%) and eGFR 55. What's first-line therapy and appropriate dose?",
   'k': 3},
  'output': {'passages': [{'id': 'pmid12345',
     'text': 'ACE inhibitor is indicated in HTN with diabetes and albuminuria. Adjust dose for eGFR <60.',
     'relevance': 0.92},
    {'id': 'pmid67890',
     'text': 'First-line for HTN in T2

In [31]:
# Test Query 1: Multi-morbidity (HTN + DM + CKD)
query1 = "58yo male, new HTN diagnosis with T2D (A1c 8.5%) and eGFR 55. What's first-line therapy and appropriate dose?"

# Test Query 2: Cardiology focus with complications
query2 = "65yo female with stable angina, HTN 150/90, started on metoprolol. Now reports fatigue and dizziness. A1c is 6.8%. Adjust therapy?"

# Test Query 3: Diabetes management with cardiovascular risk
query3 = "52yo male, T2D for 5 years, A1c 9.2%, BMI 32, history of MI 2 years ago. Currently on metformin only. What should be added?"

# Test Query 4: Complex drug interaction scenario
query4 = "Patient on lisinopril 20mg for HTN, started spironolactone for heart failure. K+ is now 5.8. What's the next step?"

# Test Query 5: Hypertension in special population
query5 = "45yo African American male, newly diagnosed HTN 165/95, no other conditions. Family history of stroke. Best initial approach?"

# Test Query 6: Acute vs chronic management
query6 = "72yo with long-standing HTN and T2D presents with BP 185/110, chest discomfort, and troponin elevation. How to manage acutely and long-term?"

In [32]:
# Run individual tests
llm = LLM(CFG)
result1 = run_collaborative_graph(llm, query1)
print_result(result1)


🚀 COLLABORATIVE MULTI-AGENT SYSTEM
Query: 58yo male, new HTN diagnosis with T2D (A1c 8.5%) and eGFR 55. What's first-line therapy and appropriate dose?

🔀 ROUTER: Analyzing query...
   Primary: cardiology|diabetes
   Consult: ['hypertension']
   Consensus needed: True

🔧 TOOLS: Gathering initial evidence...
   Executed 2 tools
   ⏭️  CARDIOLOGY: Skipped (not in routing)
   ⏭️  DIABETES: Skipped (not in routing)

🏥 HYPERTENSION SPECIALIST: Analyzing...
   🔧 Executing tools: ['search_guidelines', 'retrieve_rag']
   Confidence: 0.80
   Recommendations: 1

🤝 CONSENSUS: Synthesizing recommendations...
   Agreement level: high
   Critical alerts: 0

✅ COMPLETE


📋 ROUTING DECISION
Primary: cardiology|diabetes
Consult: ['hypertension']
Reasoning: The patient has newly diagnosed hypertension, type 2 diabetes mellitus with elevated hemoglobin A1c (8.5%), and reduced glomerular filtration rate of 55 mL/min/1.73 m². The first-line therapy for hypertension is typically a diuretic or ACE inhibitor

In [33]:
# Or run batch
test_queries = [query1, query2, query3, query4, query5, query6]
batch_results = run_batch_queries(test_queries, llm)



######################################################################
QUERY 1/6
######################################################################

🚀 COLLABORATIVE MULTI-AGENT SYSTEM
Query: 58yo male, new HTN diagnosis with T2D (A1c 8.5%) and eGFR 55. What's first-line therapy and appropriate dose?

🔀 ROUTER: Analyzing query...
   Primary: ['cardiology', 'diabetes']
   Consult: []
   Consensus needed: False

🔧 TOOLS: Gathering initial evidence...
   Executed 2 tools
   ⏭️  DIABETES: Skipped (not in routing)
   ⏭️  CARDIOLOGY: Skipped (not in routing)
   ⏭️  HYPERTENSION: Skipped (not in routing)

🤝 CONSENSUS: Synthesizing recommendations...
   Agreement level: high
   Critical alerts: 0

✅ COMPLETE



######################################################################
QUERY 2/6
######################################################################

🚀 COLLABORATIVE MULTI-AGENT SYSTEM
Query: 65yo female with stable angina, HTN 150/90, started on metoprolol. Now reports fatigue 

In [34]:
batch_results

[{'query': "58yo male, new HTN diagnosis with T2D (A1c 8.5%) and eGFR 55. What's first-line therapy and appropriate dose?",
  'result': {'user_msg': "58yo male, new HTN diagnosis with T2D (A1c 8.5%) and eGFR 55. What's first-line therapy and appropriate dose?",
   'routing': {'primary': ['cardiology', 'diabetes'],
    'consult': [],
    'requires_consensus': False,
    'urgency': 'routine',
    'reasoning': 'The patient presents with newly diagnosed hypertension, type 2 diabetes mellitus (with an A1C of 8.5%), and a reduced estimated glomerular filtration rate (eGFR of 55). The first-line therapy for both conditions is typically the same: antihypertensive medications such as ACE inhibitors or ARBs, along with lifestyle modifications to manage diabetes.'},
   'tool_results': [{'tool': 'search_guidelines',
     'args': {'query': "58yo male, new HTN diagnosis with T2D (A1c 8.5%) and eGFR 55. What's first-line therapy and appropriate dose?",
      'section': 'treatment'},
     'output': {'